# LSA-Driven Anomaly Search

This notebook actively searches for anomaly candidates using the same architecture as the pipeline:

1. Normalize raw texts into an anomaly-focused view.
2. Build TF-IDF vectors.
3. Apply LSA (Truncated SVD) to obtain dense semantic vectors.
4. Run Isolation Forest and rank documents by anomaly score.

Lower scores mean more abnormal behavior.

In [3]:
from pathlib import Path


from data_mining_assignment.core.data_io import ArticleDataset
from data_mining_assignment.tasks.anomaly_detection import TextAnomalyDetector
from data_mining_assignment.tasks.exploration import build_anomaly_candidate_table, sample_top_anomaly_texts
from data_mining_assignment.tasks.preprocessing import TextNormalizer, TextPreprocessor

## Load dataset

In [4]:
project_root_path = Path.cwd().parent
articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

document_ids = articles_data_frame["doc_id"].tolist()
raw_texts = articles_data_frame["text"].tolist()

articles_data_frame.head()

,doc_id,text
0,DOC_00001,"When I first saw this, I thought for a second ..."
1,DOC_00002,The difficulties of a high Isp OTV include: Lo...
2,DOC_00003,"Forwarded from Neal Ausman, Galileo Mission Di..."
3,DOC_00004,Sjogren's syndrome has been known to induce dr...
4,DOC_00005,"Yes, I want to concentrate on other developmen..."


## Build anomaly features with TF-IDF + LSA

In [5]:
text_normalizer = TextNormalizer()
normalized_bundle = text_normalizer.normalize_for_both_tasks(raw_texts)

anomaly_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=30000,
    min_document_frequency=1,
    max_document_frequency=1.0,
    ngram_range=(3, 5),
    analyzer_mode="char_wb",
    dense_embedding_dimension=256,
    random_seed=42,
)

anomaly_feature_matrix = anomaly_preprocessor.fit_transform(normalized_bundle.anomaly_texts)
anomaly_feature_matrix.shape

(2164, 256)

## Score anomalies

In [6]:
anomaly_detector = TextAnomalyDetector(contamination_ratio=0.02, random_seed=42)
anomaly_mask, anomaly_scores = anomaly_detector.run_detection(anomaly_feature_matrix)

anomaly_candidate_table = build_anomaly_candidate_table(
    document_ids=document_ids,
    anomaly_scores=anomaly_scores.tolist(),
    anomaly_mask=anomaly_mask.tolist(),
)

anomaly_candidate_table.head(20)

,doc_id,score,predicted_anomaly,anomaly_rank
0,DOC_01750,-0.504923,True,1
1,DOC_00909,-0.501745,True,2
2,DOC_00821,-0.482865,True,3
3,DOC_01025,-0.480890,True,4
4,DOC_00443,-0.479748,True,5
5,DOC_01211,-0.479338,True,6
6,DOC_00223,-0.478795,True,7
7,DOC_01833,-0.477093,True,8
8,DOC_02135,-0.476993,True,9
9,DOC_01114,-0.476229,True,10


## Inspect top candidates

Use this section for qualitative report evidence.

In [7]:
top_anomaly_samples = sample_top_anomaly_texts(
    anomaly_candidate_table=anomaly_candidate_table,
    document_ids=document_ids,
    raw_texts=raw_texts,
    top_k=25,
)

top_anomaly_samples[["anomaly_rank", "doc_id", "score", "predicted_anomaly", "snippet"]]

,anomaly_rank,doc_id,score,predicted_anomaly,snippet
0,1,DOC_01750,-0.504923,True,Les Bartel's comments: Let me add my .02 in. I...
1,2,DOC_00909,-0.501745,True,Les Bartel's comments: Let me add my .02 in. I...
2,3,DOC_00821,-0.482865,True,You can also swab the inside of your nose with...
3,4,DOC_01025,-0.480890,True,I need as much information about Cosmos 2238 a...
4,5,DOC_00443,-0.479748,True,Am I justified in being pissed off at this doc...
5,6,DOC_01211,-0.479338,True,Why not use the PD C library for reading/writi...
6,7,DOC_00223,-0.478795,True,Help!! I need code/package/whatever to take 3-...
7,8,DOC_01833,-0.477093,True,: Am I justified in being pissed off at this d...
8,9,DOC_02135,-0.476993,True,Other idea for old space crafts is as navigati...
9,10,DOC_01114,-0.476229,True,I recently attended an allery seminar. Steroid...


## Optional: Create submission-style top-50 table

In [8]:
submission_top_50 = anomaly_candidate_table.head(50).copy()
submission_top_50["anomaly"] = submission_top_50.index + 1
submission_top_50[["anomaly", "doc_id"]].head()

,anomaly,doc_id
0,1,DOC_01750
1,2,DOC_00909
2,3,DOC_00821
3,4,DOC_01025
4,5,DOC_00443
